# 03 - Vector Retrieval from Real Documents

## Objective

In this notebook, we will connect the components built in the
previous notebooks.

Pipeline:

PDF
↓
Text Extraction
↓
Chunking
↓
Embedding
↓
FAISS Index
↓
Query Embedding
↓
Top-K Retrieval

We will also preserve metadata such as:

- source document
- page number
- chunk ID

This metadata will later allow us to provide citations in our RAG
answers.

In [37]:
# 3. Import our own modules
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import faiss

from sentence_transformers import SentenceTransformer

# Add project root to Python path
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.loader import load_pdf
from src.chunker import create_page_chunks
# from rank_bm25 import BM25Okapi

In [2]:
# 4. Check our project structure
print("Project root:", PROJECT_ROOT)

data_dir = PROJECT_ROOT / "data"

print("\nFiles in data/:")
for file in data_dir.iterdir():
    print(file.name)

Project root: c:\Users\sansk\OneDrive\Documents\clone\ResearchRAG

Files in data/:
.gitkeep
attention-is-all-you-need-Paper.pdf
chain-of-thought.pdf
deep residual learning for image recognition.pdf
GANs.pdf
Lora.pdf


In [3]:
# 5. load multiple pdfs
pdf_files = list(data_dir.glob("*.pdf"))

print("Number of PDFs:", len(pdf_files))

for pdf in pdf_files:
    print("-", pdf.name)


Number of PDFs: 5
- attention-is-all-you-need-Paper.pdf
- chain-of-thought.pdf
- deep residual learning for image recognition.pdf
- GANs.pdf
- Lora.pdf


In [4]:
# 6. load all pdfs
all_pages = []

for pdf_path in pdf_files:
    pages = load_pdf(pdf_path)

    for page in pages:
        page["source"] = pdf_path.name

    all_pages.extend(pages)

print("Total pages:", len(all_pages))

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Type': '/Font', '/Subtype': '/Type1', '/BaseFont': '/KGMNOB+TT93o00', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 45, '/FontDescriptor': IndirectObject(64, 0, 3046514169232), '/LastChar': 121, '/Widths': [333, 0, 0, 507, 0, 507, 0, 0, 507, 507, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 440, 0, 0, 0, 440, 0, 0, 0, 0, 0, 0, 280, 0, 0, 0, 0, 0, 333, 0, 0, 0, 0, 0, 0, 480]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Type': '/Font', '/Subtype': '/Type1', '/BaseFont': '/MKDMEG+TT55o00', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 45, '/FontDescriptor': IndirectObject(312, 0, 3046514169232), '/LastChar': 121, '/Widths': [340, 0, 0, 0, 500, 0, 500, 500, 0, 0, 0, 500, 0, 0, 0, 0, 

Total pages: 72


In [5]:
# 7. Inspect the data
all_pages[0]

print("Source:", all_pages[0]["source"])
print("Page:", all_pages[0]["page"])
print("Text:", all_pages[0]["text"][:500])

Source: attention-is-all-you-need-Paper.pdf
Page: 1
Text: Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or


In [7]:
chunks = create_page_chunks(
    all_pages,
    chunk_size=1500,
    overlap_paragraphs=1
)

print("Number of chunks:", len(chunks))

Number of chunks: 76


In [8]:
for i, chunk in enumerate(chunks[:10]):

    print("=" * 100)

    print("Chunk ID:", chunk["chunk_id"])
    print("Source:", chunk["source"])
    print("Page:", chunk["page"])
    print("Characters:", len(chunk["text"]))

    print()
    print(chunk["text"][:1500])

Chunk ID: 0
Source: attention-is-all-you-need-Paper.pdf
Page: 1
Characters: 2908

Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to
be superior in quality while be

In [9]:
print("Total chunks:", len(chunks))

Total chunks: 76


In [10]:
chunks[0]

{'source': 'attention-is-all-you-need-Paper.pdf',
 'page': 1,
 'text': 'Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer,\nbased solely on attention mechanisms, dispensing with recurrence and convolutions\nentirely. Experiments on two machine translation tasks show these models to\nbe superio

In [ ]:
# 10. Why metadata matters

# This is a subtle but very important RAG design decision.

# We don't want our vector database to contain only:

# vector

# We want:

# vector
#     +
# chunk
#     +
# metadata

# Conceptually:

# ┌──────────────────────────────┐
# │ Vector                       │
# │                              │
# │ [0.21, -0.14, ...]           │
# │                              │
# │ Metadata                     │
# │ source = paper1.pdf         │
# │ page = 7                     │
# │ chunk_id = 31                │
# │                              │
# │ Text                         │
# │ "The proposed method..."     │
# └──────────────────────────────┘

# FAISS itself stores the vectors, so we maintain the metadata separately.

# This is a common pattern in vector retrieval systems.

In [29]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-base-en-v1.5")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\sansk\anaconda3\envs\rag\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sansk\.cache\huggingface\hub\models--BAAI--bge-base-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [30]:
# rebuild embeddings
chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = model.encode(
    chunk_texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

embeddings = embeddings.astype("float32")

faiss.normalize_L2(embeddings)

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

In [31]:
# rebuild faiss
embedding_dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dimension)

index.add(embeddings)

print("Embedding dimension:", embedding_dimension)
print("Vectors indexed:", index.ntotal)

Embedding dimension: 768
Vectors indexed: 76


In [32]:
def retrieve(query, model, index, chunks, k=5):

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(
        query_embedding,
        k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        result = chunks[idx].copy()
        result["score"] = float(score)

        results.append(result)

    return results

In [33]:
query = "What is self-attention?"

results = retrieve(
    query,
    model,
    index,
    chunks,
    k=10
)

for rank, result in enumerate(results, start=1):

    print("=" * 100)
    print(f"Rank: {rank}")
    print(f"Score: {result['score']:.4f}")
    print(f"Source: {result['source']}")
    print(f"Page: {result['page']}")
    print()
    print(result["text"][:1000])

Rank: 1
Score: 0.6000
Source: attention-is-all-you-need-Paper.pdf
Page: 5

MultiHead(Q,K,V ) = Concat(head 1,..., headh)W O
where headi = Attention(QW Q
i ,KW K
i ,VW V
i )
Where the projections are parameter matricesW Q
i ∈ Rdmodel×dk,W K
i ∈ Rdmodel×dk,W V
i ∈ Rdmodel×dv
andW O∈ Rhdv×dmodel.
In this work we employ h = 8 parallel attention layers, or heads. For each of these we use
dk =dv =dmodel/h = 64. Due to the reduced dimension of each head, the total computational cost
is similar to that of single-head attention with full dimensionality.
3.2.3 Applications of Attention in our Model
The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,
and the memory keys and values come from the output of the encoder. This allows every
position in the decoder to attend over all positions in the input sequence. This mimics the
typical encoder-decoder attention mechanisms in sequence-to-sequence

In [34]:
for rank, result in enumerate(results, start=1):

    print("=" * 100)

    print(f"Rank: {rank}")
    print(f"Score: {result['score']:.4f}")
    print(f"Source: {result['source']}")
    print(f"Page: {result['page']}")
    print()

    print(result["text"][:1000])

Rank: 1
Score: 0.6000
Source: attention-is-all-you-need-Paper.pdf
Page: 5

MultiHead(Q,K,V ) = Concat(head 1,..., headh)W O
where headi = Attention(QW Q
i ,KW K
i ,VW V
i )
Where the projections are parameter matricesW Q
i ∈ Rdmodel×dk,W K
i ∈ Rdmodel×dk,W V
i ∈ Rdmodel×dv
andW O∈ Rhdv×dmodel.
In this work we employ h = 8 parallel attention layers, or heads. For each of these we use
dk =dv =dmodel/h = 64. Due to the reduced dimension of each head, the total computational cost
is similar to that of single-head attention with full dimensionality.
3.2.3 Applications of Attention in our Model
The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,
and the memory keys and values come from the output of the encoder. This allows every
position in the decoder to attend over all positions in the input sequence. This mimics the
typical encoder-decoder attention mechanisms in sequence-to-sequence

In [22]:
print("Number of chunks:", len(chunks))

Number of chunks: 76


In [23]:
for i, chunk in enumerate(chunks[:5]):
    print("=" * 100)
    print("Chunk ID:", chunk["chunk_id"])
    print("Source:", chunk["source"])
    print("Page:", chunk["page"])
    print("Characters:", len(chunk["text"]))
    print()
    print(chunk["text"][:1000])

Chunk ID: 0
Source: attention-is-all-you-need-Paper.pdf
Page: 1
Characters: 2908

Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to
be superior in quality while be

In [24]:
chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = model.encode(
    chunk_texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

embeddings = embeddings.astype("float32")
faiss.normalize_L2(embeddings)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Embedding shape: (76, 384)


In [25]:
embedding_dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dimension)
index.add(embeddings)

print("Vectors indexed:", index.ntotal)

Vectors indexed: 76


In [26]:
query = "What is self-attention?"

results = retrieve(
    query,
    model,
    index,
    chunks,
    k=10
)

for rank, result in enumerate(results, start=1):
    print("=" * 100)
    print(f"Rank: {rank}")
    print(f"Score: {result['score']:.4f}")
    print(f"Source: {result['source']}")
    print(f"Page: {result['page']}")
    print()
    print(result["text"][:1000])

Rank: 1
Score: 0.3748
Source: attention-is-all-you-need-Paper.pdf
Page: 7

the input sequence centered around the respective output position. This would increase the maximum
path length toO(n/r). We plan to investigate this approach further in future work.
A single convolutional layer with kernel widthk<n does not connect all pairs of input and output
positions. Doing so requires a stack ofO(n/k) convolutional layers in the case of contiguous kernels,
orO(logk(n)) in the case of dilated convolutions [ 15], increasing the length of the longest paths
between any two positions in the network. Convolutional layers are generally more expensive than
recurrent layers, by a factor of k. Separable convolutions [ 6], however, decrease the complexity
considerably, toO(k·n·d +n·d2). Even with k = n, however, the complexity of a separable
convolution is equal to the combination of a self-attention layer and a point-wise feed-forward layer,
the approach we take in our model.
As side beneﬁt, self-att

In [27]:
for i, chunk in enumerate(chunks):

    if "self-attention is" in chunk["text"].lower():

        print("=" * 100)
        print("Chunk ID:", chunk["chunk_id"])
        print("Source:", chunk["source"])
        print("Page:", chunk["page"])
        print("Characters:", len(chunk["text"]))
        print()
        print(chunk["text"])

In [28]:
query = "What is self-attention?"

results = retrieve(
    query,
    model,
    index,
    chunks,
    k=20
)

for rank, result in enumerate(results, start=1):

    print(
        rank,
        f"{result['score']:.4f}",
        result["source"],
        "Page", result["page"],
        "Chunk", result["chunk_id"]
    )

1 0.3748 attention-is-all-you-need-Paper.pdf Page 7 Chunk 6
2 0.3431 attention-is-all-you-need-Paper.pdf Page 4 Chunk 3
3 0.3038 Lora.pdf Page 5 Chunk 54
4 0.2745 attention-is-all-you-need-Paper.pdf Page 6 Chunk 5
5 0.2722 attention-is-all-you-need-Paper.pdf Page 1 Chunk 0
6 0.2675 attention-is-all-you-need-Paper.pdf Page 5 Chunk 4
7 0.2650 attention-is-all-you-need-Paper.pdf Page 2 Chunk 1
8 0.2565 chain-of-thought.pdf Page 1 Chunk 11
9 0.2533 chain-of-thought.pdf Page 6 Chunk 16
10 0.2516 chain-of-thought.pdf Page 3 Chunk 13
11 0.2473 attention-is-all-you-need-Paper.pdf Page 11 Chunk 10
12 0.2462 chain-of-thought.pdf Page 9 Chunk 19
13 0.2418 Lora.pdf Page 9 Chunk 58
14 0.2390 chain-of-thought.pdf Page 12 Chunk 22
15 0.2379 attention-is-all-you-need-Paper.pdf Page 3 Chunk 2
16 0.2224 chain-of-thought.pdf Page 8 Chunk 18
17 0.1845 chain-of-thought.pdf Page 7 Chunk 17
18 0.1660 Lora.pdf Page 10 Chunk 59
19 0.1635 chain-of-thought.pdf Page 14 Chunk 24
20 0.1486 chain-of-thought.pdf Page

This is a much better V3 result. 🎯 But there is an important nuance.

Your new embedding model has moved the result to:

Rank 1
Page 5
Score 0.6000

However, the displayed passage is about Multi-Head Attention, not the clean definition of self-attention.

So BGE has improved the semantic retrieval, but it hasn't completely solved the question-to-evidence matching problem.

Let's compare our experiments
Version	Model	Chunking	Relevant Page 2
V1	MiniLM	Character	Rank 1
V2	MiniLM	Paragraph	Rank 7
V3	BGE	Paragraph	Not Rank 1

This is actually a valuable finding, not a failure.

It tells us:

Better embeddings ≠ guaranteed exact-answer retrieval.

In [35]:
query = "What is self-attention?"

results = retrieve(
    query,
    model,
    index,
    chunks,
    k=10
)

for rank, result in enumerate(results, start=1):

    print(
        rank,
        f"{result['score']:.4f}",
        result["source"],
        "Page", result["page"],
        "Chunk", result["chunk_id"]
    )

1 0.6000 attention-is-all-you-need-Paper.pdf Page 5 Chunk 4
2 0.5686 attention-is-all-you-need-Paper.pdf Page 4 Chunk 3
3 0.5657 attention-is-all-you-need-Paper.pdf Page 1 Chunk 0
4 0.5588 attention-is-all-you-need-Paper.pdf Page 6 Chunk 5
5 0.5343 attention-is-all-you-need-Paper.pdf Page 2 Chunk 1
6 0.5294 attention-is-all-you-need-Paper.pdf Page 3 Chunk 2
7 0.5038 chain-of-thought.pdf Page 6 Chunk 16
8 0.4955 Lora.pdf Page 10 Chunk 59
9 0.4875 Lora.pdf Page 5 Chunk 54
10 0.4854 chain-of-thought.pdf Page 7 Chunk 17


In [38]:
from rank_bm25 import BM25Okapi

In [42]:
import re


def tokenize(text):
    text = text.lower()

    # Keep words and hyphenated terms such as self-attention
    tokens = re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text)

    return tokens


tokenized_chunks = [
    tokenize(chunk["text"])
    for chunk in chunks
]

bm25 = BM25Okapi(tokenized_chunks)

print("BM25 documents:", len(tokenized_chunks))



BM25 documents: 76


In [ ]:
def bm25_retrieve(query, bm25, chunks, k=5):

    tokenized_query = tokenize(query)

    scores = bm25.get_scores(tokenized_query)

    top_indices = np.argsort(scores)[::-1][:k]

    results = []

    for idx in top_indices:

        result = chunks[idx].copy()
        result["bm25_score"] = float(scores[idx])

        results.append(result)

    return results

In [44]:
query = "What is self-attention?"

results = bm25_retrieve(
    query,
    bm25,
    chunks,
    k=10
)

for rank, result in enumerate(results, start=1):

    print("=" * 100)

    print(f"Rank: {rank}")
    print(f"BM25 Score: {result['bm25_score']:.4f}")
    print(f"Source: {result['source']}")
    print(f"Page: {result['page']}")
    print(f"Chunk: {result['chunk_id']}")

    print()
    print(result["text"][:700])

Rank: 1
BM25 Score: 6.4101
Source: Lora.pdf
Page: 10
Chunk: 59

to maximize downstream performance? 2) Is the “optimal” adaptation matrix ∆W really rank-
deﬁcient? If so, what is a good rank to use in practice? 3) What is the connection between ∆W and
W ? Does ∆W highly correlate withW ? How large is ∆W comparing toW ?
We believe that our answers to question (2) and (3) shed light on the fundamental principles of using
pre-trained language models for downstream tasks, which is a critical topic in NLP.
7.1 W HICH WEIGHT MATRICES IN TRANSFORMER SHOULD WE APPLY LORA TO?
Given a limited parameter budget, which types of weights should we adapt with LoRA to obtain
the best performance on downstream tasks? As mentioned in Section 4.2, we only consider weight
ma
Rank: 2
BM25 Score: 5.6251
Source: attention-is-all-you-need-Paper.pdf
Page: 6
Chunk: 5

Table 1: Maximum path lengths, per-layer complexity and minimum number of sequential operations
for different layer types.n is the sequence length

In [45]:
query = "What is self-attention?"

results = bm25_retrieve(
    query,
    bm25,
    chunks,
    k=30
)

for rank, result in enumerate(results, start=1):

    if (
        result["source"] == "attention-is-all-you-need-Paper.pdf"
        and result["page"] == 2
        and result["chunk_id"] == 1
    ):
        print("FOUND CORRECT CHUNK")
        print("Rank:", rank)
        print("Score:", result["bm25_score"])

FOUND CORRECT CHUNK
Rank: 3
Score: 5.122942341915649


In [46]:
def hybrid_retrieve(
    query,
    model,
    index,
    bm25,
    chunks,
    dense_k=10,
    bm25_k=10,
    final_k=5
):
    # Dense retrieval
    dense_results = retrieve(
        query,
        model,
        index,
        chunks,
        k=dense_k
    )

    # BM25 retrieval
    bm25_results = bm25_retrieve(
        query,
        bm25,
        chunks,
        k=bm25_k
    )

    # RRF scores
    rrf_scores = {}

    for rank, result in enumerate(dense_results, start=1):
        chunk_id = result["chunk_id"]
        rrf_scores[chunk_id] = rrf_scores.get(chunk_id, 0) + 1 / (60 + rank)

    for rank, result in enumerate(bm25_results, start=1):
        chunk_id = result["chunk_id"]
        rrf_scores[chunk_id] = rrf_scores.get(chunk_id, 0) + 1 / (60 + rank)

    # Sort by RRF score
    ranked_chunk_ids = sorted(
        rrf_scores,
        key=rrf_scores.get,
        reverse=True
    )[:final_k]

    # Build final results
    chunk_lookup = {
        chunk["chunk_id"]: chunk
        for chunk in chunks
    }

    results = []

    for chunk_id in ranked_chunk_ids:
        result = chunk_lookup[chunk_id].copy()
        result["rrf_score"] = rrf_scores[chunk_id]
        results.append(result)

    return results

In [47]:
query = "What is self-attention?"

results = hybrid_retrieve(
    query,
    model,
    index,
    bm25,
    chunks,
    dense_k=10,
    bm25_k=10,
    final_k=5
)

for rank, result in enumerate(results, start=1):
    print(
        f"Rank {rank} | "
        f"RRF: {result['rrf_score']:.4f} | "
        f"{result['source']} | "
        f"Page {result['page']} | "
        f"Chunk {result['chunk_id']}"
    )

Rank 1 | RRF: 0.0320 | attention-is-all-you-need-Paper.pdf | Page 5 | Chunk 4
Rank 2 | RRF: 0.0318 | attention-is-all-you-need-Paper.pdf | Page 6 | Chunk 5
Rank 3 | RRF: 0.0313 | attention-is-all-you-need-Paper.pdf | Page 2 | Chunk 1
Rank 4 | RRF: 0.0311 | Lora.pdf | Page 10 | Chunk 59
Rank 5 | RRF: 0.0305 | attention-is-all-you-need-Paper.pdf | Page 3 | Chunk 2


In [48]:
test_queries = [
    "What is self-attention?",
    "What is LoRA?",
    "What is chain of thought prompting?",
    "What is the Transformer architecture?"
]

for query in test_queries:
    results = hybrid_retrieve(
        query,
        model,
        index,
        bm25,
        chunks,
        dense_k=10,
        bm25_k=10,
        final_k=5
    )

    print(f"\nQUERY: {query}")

    for rank, result in enumerate(results, start=1):
        print(
            f"{rank}. {result['source']} | "
            f"Page {result['page']} | "
            f"Chunk {result['chunk_id']} | "
            f"RRF {result['rrf_score']:.4f}"
        )


QUERY: What is self-attention?
1. attention-is-all-you-need-Paper.pdf | Page 5 | Chunk 4 | RRF 0.0320
2. attention-is-all-you-need-Paper.pdf | Page 6 | Chunk 5 | RRF 0.0318
3. attention-is-all-you-need-Paper.pdf | Page 2 | Chunk 1 | RRF 0.0313
4. Lora.pdf | Page 10 | Chunk 59 | RRF 0.0311
5. attention-is-all-you-need-Paper.pdf | Page 3 | Chunk 2 | RRF 0.0305

QUERY: What is LoRA?
1. Lora.pdf | Page 21 | Chunk 70 | RRF 0.0325
2. Lora.pdf | Page 5 | Chunk 54 | RRF 0.0313
3. Lora.pdf | Page 2 | Chunk 51 | RRF 0.0308
4. Lora.pdf | Page 1 | Chunk 50 | RRF 0.0306
5. Lora.pdf | Page 10 | Chunk 59 | RRF 0.0164

QUERY: What is chain of thought prompting?
1. chain-of-thought.pdf | Page 6 | Chunk 16 | RRF 0.0325
2. chain-of-thought.pdf | Page 9 | Chunk 19 | RRF 0.0320
3. chain-of-thought.pdf | Page 3 | Chunk 13 | RRF 0.0320
4. chain-of-thought.pdf | Page 7 | Chunk 17 | RRF 0.0308
5. chain-of-thought.pdf | Page 1 | Chunk 11 | RRF 0.0306

QUERY: What is the Transformer architecture?
1. attention-i

---------------------------------------------------------------------

In [49]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

c:\Users\sansk\anaconda3\envs\rag\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sansk\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [50]:
def rerank(query, candidates, reranker, top_k=5):
    pairs = [
        [query, candidate["text"]]
        for candidate in candidates
    ]

    scores = reranker.predict(pairs)

    ranked = []

    for candidate, score in zip(candidates, scores):
        result = candidate.copy()
        result["rerank_score"] = float(score)
        ranked.append(result)

    ranked.sort(
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    return ranked[:top_k]

In [51]:
query = "What is self-attention?"

candidates = hybrid_retrieve(
    query,
    model,
    index,
    bm25,
    chunks,
    dense_k=10,
    bm25_k=10,
    final_k=10
)

In [52]:
reranked_results = rerank(
    query,
    candidates,
    reranker,
    top_k=5
)

In [53]:
for rank, result in enumerate(reranked_results, start=1):
    print(
        f"Rank {rank} | "
        f"Rerank Score: {result['rerank_score']:.4f} | "
        f"{result['source']} | "
        f"Page {result['page']} | "
        f"Chunk {result['chunk_id']}"
    )

Rank 1 | Rerank Score: 0.8743 | attention-is-all-you-need-Paper.pdf | Page 2 | Chunk 1
Rank 2 | Rerank Score: -0.3911 | attention-is-all-you-need-Paper.pdf | Page 6 | Chunk 5
Rank 3 | Rerank Score: -0.5393 | attention-is-all-you-need-Paper.pdf | Page 5 | Chunk 4
Rank 4 | Rerank Score: -3.5564 | attention-is-all-you-need-Paper.pdf | Page 1 | Chunk 0
Rank 5 | Rerank Score: -6.0413 | Lora.pdf | Page 10 | Chunk 59


In [54]:
for rank, result in enumerate(reranked_results, start=1):
    print(f"\n{'='*80}")
    print(f"RANK {rank}")
    print(f"SCORE: {result['rerank_score']:.4f}")
    print(f"SOURCE: {result['source']}")
    print(f"PAGE: {result['page']}")
    print(f"CHUNK: {result['chunk_id']}")
    print(f"{'='*80}")
    print(result["text"][:1000])


RANK 1
SCORE: 0.8743
SOURCE: attention-is-all-you-need-Paper.pdf
PAGE: 2
CHUNK: 1
Recurrent models typically factor computation along the symbol positions of the input and output
sequences. Aligning the positions to steps in computation time, they generate a sequence of hidden
statesht, as a function of the previous hidden stateht−1 and the input for positiont. This inherently
sequential nature precludes parallelization within training examples, which becomes critical at longer
sequence lengths, as memory constraints limit batching across examples. Recent work has achieved
signiﬁcant improvements in computational efﬁciency through factorization tricks [18] and conditional
computation [26], while also improving model performance in case of the latter. The fundamental
constraint of sequential computation, however, remains.
Attention mechanisms have become an integral part of compelling sequence modeling and transduc-
tion models in various tasks, allowing modeling of dependencies withou